# 00. 데이터 병합 및 분석 테이블 생성
### 소규모 주점 POS 원천 데이터 → 일별 분석 테이블(`origin.xlsx`)

이 노트북은 본 분석(`소규모_주점의_일매출_예측과_영업일_유형화_최종본.ipynb`)에 들어가는 **일별 분석 테이블을 원천 데이터에서 만드는 과정**입니다. POS 거래 로그를 정제해 일매출액 테이블로 집계하고 메뉴·달력·외부 데이터를 붙여 730일 × 25개 변수 테이블을 완성합니다.

| 단계 | 내용 |
|---|---|
| 1 | POS 원천 파일 불러오기·병합 |
| 2 | 품질 점검 (자료형, 중복·결측, 최댓값·최솟값, 음수, 범주) |
| 3 | 두 매출표의 총매출 대조 → 거래 로그를 기준으로 결정 |
| 4 | 거래유형별 처리 (할인, 반품·판매(반), 수량-1) |
| 5 | 일매출액 테이블과 POS 파생변수 생성 |
| 6 | 달력 변수 생성 |
| 7 | 외부 데이터 결합 (지하철 승하차, 광고·리뷰, 기상) |
| 8 | 저장 |

> **데이터 출처**
> * POS: 분석 대상 매장의 POS 기록 (2024-07-01 ~ 2026-06-30)
> * 기상(시간별 날씨·폭염·한파 특보·일평균기온): 기상청 기상자료개방포털, 서울시
> * 지하철 승하차: 서울시 지하철 호선별 역별 승하차 인원 정보 (2호선 신대방역)
> * 광고 상태·리뷰 조회수: 디너의여왕 체험단 리포트
> * 공휴일·연휴: 별도 출처 없이 달력 기준으로 직접 작성

## 1. 준비 및 POS 원천 데이터 불러오기

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from pandas import DataFrame
from IPython.display import display

from helpers import my_qtcheck, my_ts   # 자체 분석 모듈 (자료형 변환·품질 점검·시계열 인덱스)

# openpyxl 의 서식 관련 경고는 데이터와 무관하므로 숨긴다
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# 원천 데이터 위치 (이 노트북과 같은 폴더의 원본데이터/)
POS_DIR = Path('원본데이터') / 'POS'
EXT_DIR = Path('원본데이터') / '외부'
OUTPUT_PATH = Path('origin.xlsx')

📦 helpers: 데이터 분석용 자체 헬퍼 모듈 (품질 점검·전처리·통계·시각화·시계열·머신러닝·모델 진단)
📧 Email: yeoldul123@gmail.com
🐙 GitHub: https://github.com/yeoldul123
📚 참고: 아이티윌 이광호 강사님의 hossam 라이브러리(https://py.hossam.kr)를 바탕으로 만들었습니다.
🔖 Version: 0.1.0


### 1.1 POS 파일 불러오기
* **상품별 매출표** (`24~26상품별 매출.xlsx`): 2년 동안의 메뉴별 판매 합계
* **거래 로그** (`매출24_하반기` ~ `매출26_상반기`): 반기별로 나뉜 주문 단위 기록

파일 위쪽 제목 행과 중간 '합계' 행은 `번호`가 숫자가 아니므로 이를 기준으로 걸러 내고, 값이 하나도 없는 빈 컬럼은 지웁니다.

In [2]:
read_opts = dict(header=2, engine='openpyxl')
Sales_Product = pd.read_excel(POS_DIR / '24~26상품별 매출.xlsx', **read_opts)
sales_24_2 = pd.read_excel(POS_DIR / '매출24_하반기.xlsx', **read_opts)
sales_25_1 = pd.read_excel(POS_DIR / '매출25_상반기.xlsx', **read_opts)
sales_25_2 = pd.read_excel(POS_DIR / '매출25_하반기.xlsx', **read_opts)
sales_26_1 = pd.read_excel(POS_DIR / '매출26_상반기.xlsx', **read_opts)

cleaned = []
for df in [Sales_Product, sales_24_2, sales_25_1, sales_25_2, sales_26_1]:
    df = df[pd.to_numeric(df['번호'], errors='coerce').notnull()].reset_index(drop=True)  # 제목·합계 행 제거
    df = df.dropna(how='all', axis=1)                                                    # 빈 컬럼 제거
    cleaned.append(df)
Sales_Product, sales_24_2, sales_25_1, sales_25_2, sales_26_1 = cleaned

display(Sales_Product.head())
display(sales_24_2.head())

,번호,대분류,상품코드,상품명,판매수,결제 합계,개별 판매가
0,1,주류,52.0,소주,6575,32875000,5000
1,2,주류,53.0,맥주,2369,11845000,5000
2,3,안주류,42.0,순대 내장오마카세 2인,926,23150000,25000
3,4,안주류,135.0,육회,917,22925000,25000
4,5,안주류,48.0,떡볶이,726,7174000,9900


,번호,매장명,판매일시,포스,영수증,상품코드,상품명,수량,매출,타입,주문시간
0,1,대전어상,2024-12-31 23:52:47,100.0,5.0,52.0,소주,4,20000,판매,2024-12-31 23:52:44
1,2,대전어상,2024-12-31 23:52:47,100.0,5.0,50.0,김치유부우동,1,11000,판매,2024-12-31 23:26:57
2,3,대전어상,2024-12-31 23:52:47,100.0,5.0,43.0,순대 직접삶은내장,1,15000,판매,2024-12-31 23:26:52
3,4,대전어상,2024-12-31 23:52:47,100.0,5.0,52.0,소주,1,5000,판매,2024-12-31 20:36:15
4,5,대전어상,2024-12-31 23:52:47,100.0,5.0,135.0,육회,1,25000,판매,2024-12-31 20:36:14


### 1.2 반기별 거래 로그 병합

In [3]:
# 반기 파일마다 붙은 행 번호(번호)는 병합 후 의미가 없으므로 지운다
logs = [df.drop(columns=['번호']) if '번호' in df.columns else df
        for df in [sales_24_2, sales_25_1, sales_25_2, sales_26_1]]
total_sales = pd.concat(logs, ignore_index=True)

display(total_sales.head())
print(f"통합 거래 로그 크기 - 행: {total_sales.shape[0]}, 열: {total_sales.shape[1]}")

,매장명,판매일시,포스,영수증,상품코드,상품명,수량,매출,타입,주문시간
0,대전어상,2024-12-31 23:52:47,100.0,5.0,52.0,소주,4,20000,판매,2024-12-31 23:52:44
1,대전어상,2024-12-31 23:52:47,100.0,5.0,50.0,김치유부우동,1,11000,판매,2024-12-31 23:26:57
2,대전어상,2024-12-31 23:52:47,100.0,5.0,43.0,순대 직접삶은내장,1,15000,판매,2024-12-31 23:26:52
3,대전어상,2024-12-31 23:52:47,100.0,5.0,52.0,소주,1,5000,판매,2024-12-31 20:36:15
4,대전어상,2024-12-31 23:52:47,100.0,5.0,135.0,육회,1,25000,판매,2024-12-31 20:36:14


통합 거래 로그 크기 - 행: 15222, 열: 10


## 2. 품질 점검

### 2.1 컬럼 정리와 자료형 변환
* `매장명`, `포스`는 모든 행이 같은 값이라 지웁니다.
* 이름을 뜻이 바로 보이게 바꿉니다: `영수증` → `방문 테이블 수`, `타입` → `거래유형`, `매출` → `가격`
* 상품별 매출표의 `개별 판매가`에 숫자가 아닌 값이 들어 있는 행을 찾아 확인합니다.

In [4]:
total_sales = total_sales.drop(columns=['매장명', '포스'])
Sales_Product = Sales_Product.drop(columns=['번호'])
total_sales = total_sales.rename(columns={'영수증': '방문 테이블 수', '타입': '거래유형', '매출': '가격'})

total_sales = my_qtcheck.set_type(total_sales, as_int=['방문 테이블 수'], as_float=[],
                                  as_string=['상품명', '상품코드'], as_category=['거래유형'],
                                  as_datetime=['판매일시', '주문시간'])

non_numeric_rows = Sales_Product[pd.to_numeric(Sales_Product['개별 판매가'], errors='coerce').isna()]
non_numeric_rows

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15222 entries, 0 to 15221
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   판매일시      15222 non-null  datetime64[ns]
 1   방문 테이블 수  15222 non-null  int64         
 2   상품코드      15222 non-null  object        
 3   상품명       15222 non-null  object        
 4   수량        15222 non-null  int64         
 5   가격        15222 non-null  int64         
 6   거래유형      15222 non-null  category      
 7   주문시간      15222 non-null  datetime64[ns]
dtypes: category(1), datetime64[ns](2), int64(3), object(2)
memory usage: 847.6+ KB


,대분류,상품코드,상품명,판매수,결제 합계,개별 판매가
316,수제튀김,9.0,대왕 오징어 (4개),0,0,#DIV/0!


판매 수가 0인 행이라 매출에 영향이 없으므로 지웁니다. 두 테이블을 나중에 맞대어 보기 위해 거래 로그에서도 같은 상품코드의 행을 확인합니다.

In [5]:
Sales_Product = Sales_Product.drop(non_numeric_rows.index)

# 상품코드는 문자열이므로 '9.0' 으로 비교한다
target_rows = total_sales[total_sales['상품코드'] == '9.0']
display(target_rows)

,판매일시,방문 테이블 수,상품코드,상품명,수량,가격,거래유형,주문시간
3087,2024-07-02 18:55:44,2,9.0,대왕 오징어 (4개),-5,-30000,반품,2024-07-02 18:55:17
3092,2024-07-02 18:55:21,1,9.0,대왕 오징어 (4개),5,30000,판매(반),2024-07-02 18:55:17


두 행 모두 실제 반품 상품이라 지워도 전체 매출에는 영향이 없습니다. 지웁니다.

In [6]:
total_sales = total_sales.drop(index=target_rows.index)
print(f"남은 행 수: {len(total_sales)}")

Sales_Product = my_qtcheck.set_type(Sales_Product, as_int=['개별 판매가'], as_float=[],
                                    as_string=['상품코드', '상품명', '대분류'], as_category=[], as_datetime=[])

남은 행 수: 15220
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 316 entries, 0 to 315
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   대분류     316 non-null    object
 1   상품코드    316 non-null    object
 2   상품명     316 non-null    object
 3   판매수     316 non-null    int64 
 4   결제 합계   316 non-null    int64 
 5   개별 판매가  316 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 14.9+ KB


### 2.2 중복과 결측 확인

In [7]:
print('[거래 로그]')
my_qtcheck.check_duplicates(total_sales, drop=False)
display(my_qtcheck.check_missing_values(total_sales))

print('[상품별 매출표]')
my_qtcheck.check_duplicates(Sales_Product, drop=False)
display(my_qtcheck.check_missing_values(Sales_Product))

[거래 로그]
중복된 행의 수: 0


,Missing Count,Missing Ratio (%)
판매일시,0,0.0
방문 테이블 수,0,0.0
상품코드,0,0.0
상품명,0,0.0
수량,0,0.0
가격,0,0.0
거래유형,0,0.0
주문시간,0,0.0


[상품별 매출표]
중복된 행의 수: 0


,Missing Count,Missing Ratio (%)
대분류,0,0.0
상품코드,0,0.0
상품명,0,0.0
판매수,0,0.0
결제 합계,0,0.0
개별 판매가,0,0.0


### 2.3 최댓값·최솟값 점검

In [8]:
def minmax_table(df):
    """수치형·날짜형 컬럼의 최솟값과 최댓값을 표로 만든다."""
    fields = df.select_dtypes(include=['number', 'datetime']).columns.tolist()
    return DataFrame([{'min': df[f].min(), 'max': df[f].max()} for f in fields], index=fields)

display(minmax_table(total_sales))
display(minmax_table(Sales_Product))

,min,max
판매일시,2024-07-02 18:55:21,2026-06-30 22:39:58
방문 테이블 수,1,18
수량,-3,27
가격,-59000,171000
주문시간,2024-07-02 18:55:07,2026-06-30 22:39:56


,min,max
판매수,1,6575
결제 합계,-50000,32875000
개별 판매가,-10000,99000


두 테이블 모두 음수가 있습니다. 거래 로그는 `수량`과 `가격`, 상품별 매출표는 `결제 합계`와 `개별 판매가`입니다. 어떤 행인지 확인합니다.

In [9]:
display(total_sales[(total_sales['수량'] < 0) | (total_sales['가격'] < 0)])
display(Sales_Product[(Sales_Product['결제 합계'] < 0) | (Sales_Product['개별 판매가'] < 0)])

,판매일시,방문 테이블 수,상품코드,상품명,수량,가격,거래유형,주문시간
168,2024-12-24 01:11:26,2,82.0,할인,1,-2000,판매,2024-12-24 01:11:24
293,2024-12-18 01:39:51,2,43.0,순대 직접삶은내장,-1,-15000,반품,2024-12-18 01:38:33
515,2024-12-01 22:22:33,6,82.0,할인,1,-2000,판매,2024-12-01 22:22:07
523,2024-12-01 00:31:48,2,83.0,포장할인,1,-1000,판매,2024-12-01 00:31:47
581,2024-11-28 22:27:22,4,83.0,포장할인,1,-1000,판매,2024-11-28 22:27:20
...,...,...,...,...,...,...,...,...
14008,2026-02-28 23:24:19,9,53.0,맥주,-1,-5000,반품,2026-02-28 23:20:34
14009,2026-02-28 23:24:19,9,52.0,소주,-3,-15000,반품,2026-02-28 23:20:33
14010,2026-02-28 23:24:19,9,330.0,해창막걸리,-1,-19000,반품,2026-02-28 23:20:30
14011,2026-02-28 23:24:19,9,49.0,가마보꼬우동,-1,-12000,반품,2026-02-28 23:20:11


,대분류,상품코드,상품명,판매수,결제 합계,개별 판매가
47,안주류,83.0,포장할인,48,-48000,-1000
79,안주류,82.0,할인,25,-50000,-2000
310,안주류,271.0,핵심점포 이벤트,1,-10000,-10000


* **거래 로그**: 거래유형이 반품인 행은 수량과 가격이 **둘 다 음수**입니다. 판매는 대체로 수량이 양수라 수량 × 가격으로 매출을 구하면 할인 등이 자연스럽게 빠지지만 반품은 음수끼리 곱해져 **양수가 되므로 매출이 부풀려질 위험**이 있습니다. 4단계에서 따로 처리합니다.
* **상품별 매출표**: 포장할인, 할인, 핵심점포 이벤트 같은 할인 항목입니다.
* 둘 다 잘못 기록된 값은 아닙니다.

### 2.4 범주형 컬럼 분포

In [10]:
def display_column_proportions(df, df_name):
    """범주형 컬럼마다 범주별 빈도와 비율을 표로 출력한다."""
    print(f"=== [{df_name}] 컬럼별 범주 요약 ===")
    for col in df.select_dtypes(include=['object', 'category']).columns:
        counts = df[col].value_counts(dropna=False)
        proportions = df[col].value_counts(normalize=True, dropna=False)
        print(f"\n[컬럼: {col}]")
        display(DataFrame({'범주값': counts.index, '집단 수(빈도)': counts.values,
                           '전체 대비 비율': proportions.values}))

display_column_proportions(total_sales, '거래 로그')
display_column_proportions(Sales_Product, '상품별 매출표')

=== [거래 로그] 컬럼별 범주 요약 ===

[컬럼: 상품코드]


,범주값,집단 수(빈도),전체 대비 비율
0,52.0,2789,0.183246
1,53.0,1172,0.077004
2,42.0,912,0.059921
3,135.0,903,0.059330
4,48.0,713,0.046846
...,...,...,...
311,75.0,1,0.000066
312,316.0,1,0.000066
313,71.0,1,0.000066
314,70.0,1,0.000066



[컬럼: 상품명]


,범주값,집단 수(빈도),전체 대비 비율
0,소주,2789,0.183246
1,맥주,1172,0.077004
2,육회,905,0.059461
3,떡볶이,713,0.046846
4,김치유부우동,514,0.033771
...,...,...,...
320,두두물물약주,1,0.000066
321,풍정사계 춘,1,0.000066
322,황금보리19,1,0.000066
323,라면볶이,1,0.000066



[컬럼: 거래유형]


...

,범주값,집단 수(빈도),전체 대비 비율
0,증류식소주,59,0.186709
1,안주류,58,0.183544
2,약주청주,48,0.151899
3,주류,41,0.129747
4,막걸리,41,0.129747
5,하이볼,20,0.063291
6,조선위스키,14,0.044304
7,옛날 추억의 분식,8,0.025316
8,과실주,6,0.018987
9,음료수,5,0.015823



[컬럼: 상품코드]


,범주값,집단 수(빈도),전체 대비 비율
0,52.0,1,0.003165
1,25.0,1,0.003165
2,62.0,1,0.003165
3,319.0,1,0.003165
4,341.0,1,0.003165
...,...,...,...
311,81.0,1,0.003165
312,228.0,1,0.003165
313,222.0,1,0.003165
314,309.0,1,0.003165



[컬럼: 상품명]


,범주값,집단 수(빈도),전체 대비 비율
0,왕주25,3,0.009494
1,매운우동,2,0.006329
2,김포예주,2,0.006329
3,오절판 다과상,2,0.006329
4,사과하이볼,2,0.006329
...,...,...,...
291,시원칼칼한 꼬치오뎅 1개,1,0.003165
292,시원칼칼한 꼬치오뎅 3개,1,0.003165
293,우렁이청주,1,0.003165
294,필25,1,0.003165


* **거래 로그**
  * `상품명`: 326개. 범주가 많고 판매 횟수가 고르지 않습니다(한 번만 팔린 상품도 있음).
  * `거래유형`: 판매 외에 반품, 판매(반), 수량-1이 있지만 모두 1% 미만입니다. 판매(반)과 반품의 행 수가 같습니다.
* **상품별 매출표**
  * `대분류`: 15개
  * `상품명`: 296개로 거래 로그보다 약 30개 적고 상품코드가 없습니다. 나중에 두 테이블을 이어 붙인다면 상품명을 기준으로 삼아야 합니다.

## 3. 두 매출표의 총매출 대조

병합하기 전에 두 테이블의 총매출이 맞는지 확인합니다.

In [11]:
sales_product_revenue = Sales_Product['결제 합계'].sum()
total_sales_revenue = (total_sales['수량'] * total_sales['가격']).sum()   # 반품 등 예외 행(1% 미만) 포함

print(f"상품별 매출표 총매출: {sales_product_revenue:,}원")
print(f"거래 로그 총매출:     {total_sales_revenue:,}원")
print(f"차이:                {abs(total_sales_revenue - sales_product_revenue):,}원")

상품별 매출표 총매출: 240,978,800원
거래 로그 총매출:     413,038,800원
차이:                172,060,000원


차이가 상당히 큽니다. 어디서 생기는지 보려고 거래 로그로 메뉴별 매출을 다시 집계해 상품별 매출표와 나란히 비교합니다.

In [12]:
total_agg = (total_sales.assign(금액=total_sales['수량'] * total_sales['가격'])
             .groupby('상품명')['금액'].sum()
             .reset_index(name='거래로그_금액'))
product_agg = Sales_Product[['상품명', '결제 합계']].rename(columns={'결제 합계': '상품별매출표_금액'})

comparison_df = pd.merge(total_agg, product_agg, on='상품명', how='outer').fillna(0)
comparison_df['매출_차이'] = comparison_df['거래로그_금액'] - comparison_df['상품별매출표_금액']
display(comparison_df.sort_values(by='매출_차이', ascending=False))

,상품명,거래로그_금액,상품별매출표_금액,매출_차이
158,소주,129185000,32875000.0,96310000.0
88,맥주,40105000,11845000.0,28260000.0
246,육회,25178000,133000.0,25045000.0
178,순대 내장볶음小,13125000,0.0,13125000.0
175,순대 내장볶음+삶은 간 2인,8125000,0.0,8125000.0
...,...,...,...,...
298,치마살 육회,679000,1344000.0,-665000.0
255,이도42,65000,806000.0,-741000.0
33,내장볶음1인,12000,984000.0,-972000.0
181,순대 내장오마카세 4인,305000,6430000.0,-6125000.0


메뉴 단위에서도 두 테이블이 서로 맞지 않습니다. 상품별 매출표는 이미 합산된 결과라 원인을 되짚을 수 없으므로, **주문 하나하나가 남아 있는 원본 거래 로그만 믿고** 이후 집계를 진행합니다.

In [13]:
# 점검을 마친 거래 로그를 중간 결과로 저장
total_sales.to_excel('total_sales_qtcheck.xlsx', index=False)

## 4. 거래유형별 처리

일매출을 정확히 집계하려면 수량이나 가격이 음수·0인 행을 거래유형별로 확인해야 합니다. 거래유형별로 수량·가격의 범위와 부호 조합을 봅니다.

In [14]:
def check_signs(row):
    """수량과 가격의 부호 조합을 문자열로 돌려준다."""
    q, p = row['수량'], row['가격']
    if q < 0 and p < 0: return '--'
    elif q > 0 and p > 0: return '++'
    elif q < 0 and p > 0: return '-+'
    elif q > 0 and p < 0: return '+-'
    else: return '0 포함'

print('--- 판매: 수량/가격 범위 ---')
display(total_sales[total_sales['거래유형'] == '판매'][['수량', '가격']].agg(['min', 'max']))

for t in ['반품', '판매(반)']:
    part = total_sales[total_sales['거래유형'] == t]
    print(f'\n--- {t}: 수량/가격 범위 ---')
    display(part[['수량', '가격']].agg(['min', 'max']))
    signs = part.apply(check_signs, axis=1).value_counts().reset_index()
    signs.columns = ['부호상태', '건수']
    print(f'--- {t}: 부호 상태 분포 ---')
    display(signs)

print('\n--- 수량-1 행 ---')
display(total_sales[total_sales['거래유형'] == '수량-1'])

--- 판매: 수량/가격 범위 ---


,수량,가격
min,1,-10000
max,27,171000



--- 반품: 수량/가격 범위 ---


,수량,가격
min,-3,-59000
max,-1,2000


--- 반품: 부호 상태 분포 ---


...

,수량,가격
min,1,-2000
max,3,59000


--- 판매(반): 부호 상태 분포 ---


,부호상태,건수
0,++,54
1,+-,1



--- 수량-1 행 ---


,판매일시,방문 테이블 수,상품코드,상품명,수량,가격,거래유형,주문시간
6979,2025-11-28 22:03:03,8,31.0,코카콜라캔 355ml,-1,-2500,수량-1,2025-11-28 21:51:52
7387,2025-11-09 22:43:57,7,135.0,육회,-1,-25000,수량-1,2025-11-09 20:36:27
7843,2025-10-26 03:07:20,4,31.0,코카콜라캔 355ml,-1,-2500,수량-1,2025-10-25 22:37:40
8970,2025-09-14 23:34:02,8,252.0,안동소주하이볼,-1,-10000,수량-1,2025-09-14 20:45:53
13328,2026-04-04 19:40:43,4,53.0,맥주,-1,-5000,수량-1,2026-04-04 17:51:10


### 4.1 판매 유형의 음수·0 행
판매 유형에도 수량이나 가격이 0 이하인 행이 있습니다. 무엇인지 직접 확인합니다.

In [15]:
sale_abnormal_df = total_sales[(total_sales['거래유형'] == '판매') &
                               ((total_sales['수량'] <= 0) | (total_sales['가격'] <= 0))]
display(sale_abnormal_df)

product_dist = sale_abnormal_df['상품명'].value_counts().reset_index()
product_dist.columns = ['상품명', '건수']
display(product_dist)

,판매일시,방문 테이블 수,상품코드,상품명,수량,가격,거래유형,주문시간
168,2024-12-24 01:11:26,2,82.0,할인,1,-2000,판매,2024-12-24 01:11:24
515,2024-12-01 22:22:33,6,82.0,할인,1,-2000,판매,2024-12-01 22:22:07
523,2024-12-01 00:31:48,2,83.0,포장할인,1,-1000,판매,2024-12-01 00:31:47
581,2024-11-28 22:27:22,4,83.0,포장할인,1,-1000,판매,2024-11-28 22:27:20
667,2024-11-23 23:39:59,11,83.0,포장할인,1,-1000,판매,2024-11-23 23:39:57
934,2024-11-10 20:06:23,9,83.0,포장할인,1,-1000,판매,2024-11-10 20:06:19
962,2024-11-09 18:41:24,3,83.0,포장할인,1,-1000,판매,2024-11-09 18:41:21
975,2024-11-08 22:58:01,4,83.0,포장할인,1,-1000,판매,2024-11-08 22:57:55
1030,2024-11-05 18:25:14,1,83.0,포장할인,1,-1000,판매,2024-11-05 18:25:12
1132,2024-10-31 23:43:01,5,83.0,포장할인,1,-1000,판매,2024-10-31 23:42:58


,상품명,건수
0,포장할인,41
1,할인,24
2,핵심점포 이벤트,1


66행 모두 할인 항목입니다. 수량은 전부 1 이상이고 가격이 음수 또는 0이라, **수량 × 가격으로 더하면 할인만큼 매출이 줄어 그대로 두어도 집계에 문제가 없습니다.**

### 4.2 반품과 판매(반)
* **반품**: 수량은 모두 음수인데 가격에는 양수도 있어 부호가 엇갈리는 행과 같은 행이 섞여 있습니다.
* **판매(반)**: 반대로 수량은 모두 양수이고 가격에 음수가 있습니다.

두 유형이 짝을 이루는지 확인합니다. **상품코드가 같고 가격의 절댓값이 같으며 거래유형이 서로 다른 두 행**을 한 쌍으로 지워 나가고, 짝이 없는 행이 남는지 봅니다.

In [16]:
return_df = total_sales[total_sales['거래유형'].isin(['반품', '판매(반)'])].copy()
return_df['절대가격'] = return_df['가격'].abs()   # 부호를 무시하고 비교

drop_indices, matched = [], set()
for idx1, row1 in return_df.iterrows():
    if idx1 in matched:
        continue
    match = return_df[(return_df.index != idx1) &
                      (~return_df.index.isin(matched)) &
                      (return_df['상품코드'] == row1['상품코드']) &
                      (return_df['절대가격'] == row1['절대가격']) &
                      (return_df['거래유형'] != row1['거래유형'])]
    if not match.empty:
        idx2 = match.index[0]
        drop_indices += [idx1, idx2]
        matched.update([idx1, idx2])

remaining_df = return_df.drop(index=drop_indices)
print(f"짝을 이뤄 제거된 행 수: {len(drop_indices)}개")
print('--- 짝이 없는 행 ---')
display(remaining_df)

짝을 이뤄 제거된 행 수: 110개
--- 짝이 없는 행 ---


,판매일시,방문 테이블 수,상품코드,상품명,수량,가격,거래유형,주문시간,절대가격


모든 행이 짝을 이룹니다. 반품·판매(반)로 잘못 기록된 행은 없습니다.

다만 이 행들을 수량 × 가격으로 집계하면 55개 행이 모두 양수가 되어 매출이 두 번 더해집니다. 전체 행의 1% 미만이므로 **매출 집계에서 반품·판매(반)을 빼기로** 했습니다. `수량-1`도 실물 재고를 맞추려고 입력한 것으로 보이고 매출에 미치는 영향이 작아 함께 뺍니다. 결과적으로 **거래유형이 '판매'인 행만** 집계합니다.

## 5. 일매출액 테이블과 POS 파생변수

### 5.1 일매출액 집계

In [17]:
df_sale = total_sales[total_sales['거래유형'] == '판매'].copy()
print(f"판매 행 수: {len(df_sale)}")

df_sale['영업일'] = pd.to_datetime(df_sale['판매일시']).dt.date      # 달력 날짜 기준 영업일
df_sale['매출액'] = df_sale['수량'] * df_sale['가격']

daily_revenue = df_sale.groupby('영업일')['매출액'].sum().reset_index()
display(daily_revenue)

판매 행 수: 15105


,영업일,매출액
0,2024-07-03,124000
1,2024-07-04,28000
2,2024-07-05,432500
3,2024-07-06,360000
4,2024-07-07,294000
...,...,...
703,2026-06-26,1924500
704,2026-06-27,401000
705,2026-06-28,412500
706,2026-06-29,102000


In [18]:
# 관측 기간 전체 날짜를 만들고 매출이 없는 날(비영업일)을 찾는다
full_date_range = pd.date_range(start='2024-07-01', end='2026-06-30', freq='D')
check = daily_revenue.set_index('영업일').reindex(full_date_range)
missing_dates = check[check['매출액'].isna()].index
print(f"--- 매출이 없는 날 (총 {len(missing_dates)}일) ---")
print(missing_dates.date)

--- 매출이 없는 날 (총 22일) ---
[datetime.date(2024, 7, 1) datetime.date(2024, 7, 2)
 datetime.date(2024, 7, 30) datetime.date(2024, 8, 12)
 datetime.date(2024, 8, 13) datetime.date(2024, 12, 9)
 datetime.date(2025, 3, 17) datetime.date(2025, 5, 11)
 datetime.date(2025, 5, 12) datetime.date(2025, 5, 13)
 datetime.date(2025, 5, 14) datetime.date(2025, 5, 15)
 datetime.date(2025, 5, 16) datetime.date(2025, 5, 17)
 datetime.date(2025, 5, 18) datetime.date(2025, 5, 19)
 datetime.date(2025, 11, 17) datetime.date(2026, 1, 26)
 datetime.date(2026, 3, 11) datetime.date(2026, 4, 7)
 datetime.date(2026, 4, 20) datetime.date(2026, 6, 15)]


매출이 없는 날이 22일 있습니다. 시계열 분석에서는 영업이 끊기는 지점이 중요하므로, 이 날들은 지우지 않고 6장에서 휴무일로 표시합니다.

### 5.2 메뉴 카테고리 결합
카테고리별 판매건수를 만들기 위해 메뉴 테이블의 카테고리를 상품코드로 붙입니다. 먼저 두 테이블의 코드가 서로 빠짐없이 대응하는지 확인합니다.

In [19]:
menu = pd.read_excel(POS_DIR / '메뉴테이블(값대체).xlsx')[['메뉴_ID', '메뉴명', '카테고리']].copy()

# 소수점(.0)을 떼고 문자열로 맞춰 차집합을 비교한다
sales_codes = set(df_sale['상품코드'].astype(str).str.replace('.0', '', regex=False).dropna())
menu_ids = set(menu['메뉴_ID'].astype(str).str.replace('.0', '', regex=False).dropna())
print(f"메뉴 테이블에만 있는 메뉴_ID: {menu_ids - sales_codes}")
print(f"거래 로그에만 있는 상품코드: {sales_codes - menu_ids}")

메뉴 테이블에만 있는 메뉴_ID: {'9'}
거래 로그에만 있는 상품코드: {'213'}


In [20]:
df_sale['상품코드'] = df_sale['상품코드'].astype(float).astype(int)
display(df_sale[df_sale['상품코드'] == 213])

,판매일시,방문 테이블 수,상품코드,상품명,수량,가격,거래유형,주문시간,영업일,매출액
4721,2025-03-07 21:21:52,7,213,콜키지,1,20000,판매,2025-03-07 20:41:54,2025-03-07,20000
8743,2025-09-26 20:13:57,4,213,콜키지,1,20000,판매,2025-09-26 20:06:51,2025-09-26,20000
10798,2025-06-27 23:02:35,10,213,콜키지,1,20000,판매,2025-06-27 22:25:22,2025-06-27,20000
10905,2025-06-21 20:40:45,4,213,콜키지,1,20000,판매,2025-06-21 19:22:55,2025-06-21,20000
11902,2026-06-07 01:04:57,2,213,콜키지(병당),1,20000,판매,2026-06-07 00:09:12,2026-06-07,20000
14563,2026-02-06 20:48:04,6,213,콜키지(병당),3,60000,판매,2026-02-06 20:48:01,2026-02-06,180000


상품코드 213은 **콜키지**입니다. 주류에 속하지만 매장 재고가 아닌 외부 주류라 카테고리별 판매건수에서는 뺍니다. 메뉴 테이블에 213이 없으므로 내부 결합(`inner`)으로 붙이면 자연히 빠집니다. 다만 매출액·피크집중도·고유판매 메뉴수는 콜키지까지 포함한 `df_sale` 기준으로 집계합니다.

In [21]:
menu['메뉴_ID'] = menu['메뉴_ID'].astype(float).astype(int)
merged_df = pd.merge(df_sale, menu, left_on='상품코드', right_on='메뉴_ID', how='inner')
print(f"결합 전 {len(df_sale)}행 → 결합 후 {len(merged_df)}행 (콜키지 행만큼 감소)")

결합 전 15105행 → 결합 후 15099행 (콜키지 행만큼 감소)


### 5.3 카테고리별 판매건수와 영수증수

In [22]:
daily_table = merged_df[['영업일']].drop_duplicates().sort_values('영업일').reset_index(drop=True)

counts = {
    '메인메뉴 판매건수': merged_df[merged_df['카테고리'] == 'M(메인)'],
    '전통주류 판매건수': merged_df[merged_df['카테고리'] == 'T(전통주)'],
    '일반주류 판매건수': merged_df[merged_df['카테고리'] == 'A(일반주류)'],
    '전체메뉴 판매건수': merged_df,
}
final_table = daily_table
for name, part in counts.items():
    final_table = final_table.merge(part.groupby('영업일')['수량'].sum().reset_index(name=name),
                                    on='영업일', how='left')
final_table = final_table.fillna(0)          # 해당 카테고리 판매가 없는 날은 0

# 영수증수: 영업일별 판매일시(결제 단위)의 고유 개수
receipts = merged_df.groupby('영업일', as_index=False).agg(영수증수=('판매일시', 'nunique'))

origin = pd.merge(daily_revenue, final_table, on='영업일', how='inner')
origin = pd.merge(origin, receipts, on='영업일', how='inner')
display(origin.head())

,영업일,매출액,메인메뉴 판매건수,전통주류 판매건수,일반주류 판매건수,전체메뉴 판매건수,영수증수
0,2024-07-03,124000,0.0,3.0,0.0,13,3
1,2024-07-04,28000,0.0,2.0,0.0,4,2
2,2024-07-05,432500,1.0,8.0,4.0,34,8
3,2024-07-06,360000,2.0,3.0,13.0,30,5
4,2024-07-07,294000,1.0,4.0,8.0,27,5


### 5.4 평균 객단가, 피크집중도, 고유판매 메뉴수

In [23]:
# 평균 객단가 = 매출액 / 영수증수
origin['평균 객단가'] = (origin['매출액'] / origin['영수증수']).round(0)

# 피크집중도 = 첫 주문이 19~21시인 영수증 비율
#   손님이 언제 몰리는지가 핵심이므로 영수증마다 첫 주문 시각을 기준으로 삼는다 (콜키지 포함, df_sale 기준)
df_sale['첫주문시간'] = df_sale.groupby('판매일시')['주문시간'].transform('min')
receipt_summary = df_sale[['영업일', '판매일시', '첫주문시간']].drop_duplicates()
total_receipts = receipt_summary.groupby('영업일')['판매일시'].nunique()
peak_receipts = receipt_summary[(receipt_summary['첫주문시간'].dt.hour >= 19) &
                                (receipt_summary['첫주문시간'].dt.hour < 21)
                                ].groupby('영업일')['판매일시'].nunique()
peak = (peak_receipts / total_receipts).fillna(0).reset_index(name='피크집중도')   # 피크 주문이 없으면 0
origin = pd.merge(origin, peak, on='영업일', how='left')

# 고유판매 메뉴수 = 그날 한 번이라도 팔린 메뉴 종류 수
unique_items = df_sale.groupby('영업일')['상품코드'].nunique().reset_index(name='고유판매 메뉴수')
origin = pd.merge(origin, unique_items, on='영업일', how='left')
origin.head()

,영업일,매출액,메인메뉴 판매건수,전통주류 판매건수,일반주류 판매건수,전체메뉴 판매건수,영수증수,평균 객단가,피크집중도,고유판매 메뉴수
0,2024-07-03,124000,0.0,3.0,0.0,13,3,41333.0,0.00,5
1,2024-07-04,28000,0.0,2.0,0.0,4,2,14000.0,0.00,3
2,2024-07-05,432500,1.0,8.0,4.0,34,8,54062.0,0.75,15
3,2024-07-06,360000,2.0,3.0,13.0,30,5,72000.0,0.40,15
4,2024-07-07,294000,1.0,4.0,8.0,27,5,58800.0,0.20,13


## 6. 달력 변수

### 6.1 관측 기간 전체로 날짜 채우기
거래 로그의 첫 영업일은 7월 3일이라, 관측 시작일인 7월 1·2일 행을 앞에 추가한 뒤 하루 간격 시계열 인덱스로 맞춥니다. 매출이 없는 날은 결측으로 남습니다.

In [24]:
extra_rows = pd.DataFrame({'영업일': ['2024-07-01', '2024-07-02']})
origin = pd.concat([extra_rows, origin], ignore_index=True)
origin = my_ts.set_index(origin, column='영업일', freq='D').reset_index()
origin.head()

,영업일,매출액,메인메뉴 판매건수,전통주류 판매건수,일반주류 판매건수,전체메뉴 판매건수,영수증수,평균 객단가,피크집중도,고유판매 메뉴수
0,2024-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-07-03,124000.0,0.0,3.0,0.0,13.0,3.0,41333.0,0.00,5.0
3,2024-07-04,28000.0,0.0,2.0,0.0,4.0,2.0,14000.0,0.00,3.0
4,2024-07-05,432500.0,1.0,8.0,4.0,34.0,8.0,54062.0,0.75,15.0


### 6.2 연속영업일수와 휴무여부

In [25]:
is_business = origin['매출액'].notna()            # 매출이 있으면 영업일
group_id = (~is_business).cumsum()                 # 휴무일을 만날 때마다 구간 번호 증가

origin['연속영업일수'] = 0
origin.loc[is_business, '연속영업일수'] = origin[is_business].groupby(group_id).cumcount() + 1
origin['휴무여부'] = origin['매출액'].isna().astype(int)

### 6.3 공휴일, 공휴일 전날, 요일, 계절, 연휴
공휴일과 연휴는 별도 출처 없이 달력을 기준으로 직접 정리했습니다. 연휴는 2일 이상 이어지는 명절·장기 연휴입니다.

In [26]:
all_holidays = [
    # 2024년
    '2024-08-15', '2024-09-16', '2024-09-17', '2024-09-18', '2024-10-03', '2024-10-09', '2024-12-25',
    # 2025년
    '2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-03-01', '2025-03-03',
    '2025-05-05', '2025-05-06', '2025-06-06', '2025-08-15', '2025-10-03', '2025-10-06', '2025-10-07',
    '2025-10-08', '2025-10-09', '2025-12-25',
    # 2026년
    '2026-01-01', '2026-02-16', '2026-02-17', '2026-02-18', '2026-03-01', '2026-03-02', '2026-05-05',
    '2026-05-24', '2026-05-25', '2026-06-03', '2026-06-06',
]

holiday_periods = [
    '2024-09-16', '2024-09-17', '2024-09-18',                       # 2024 추석
    '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30',         # 2025 설날 (임시공휴일 포함)
    '2025-03-01', '2025-03-03',                                     # 2025 삼일절 (대체공휴일 포함)
    '2025-05-05', '2025-05-06',                                     # 2025 어린이날·부처님오신날
    '2025-10-03', '2025-10-06', '2025-10-07', '2025-10-08', '2025-10-09',  # 2025 개천절·추석·한글날
    '2026-02-16', '2026-02-17', '2026-02-18',                       # 2026 설날
    '2026-03-01', '2026-03-02',                                     # 2026 삼일절 (대체공휴일 포함)
    '2026-05-24', '2026-05-25',                                     # 2026 부처님오신날 (대체공휴일 포함)
]

def get_season(date):
    """월로 계절을 정한다 (봄 3~5월, 여름 6~8월, 가을 9~11월, 겨울 12~2월)."""
    if date.month in [3, 4, 5]: return '봄'
    if date.month in [6, 7, 8]: return '여름'
    if date.month in [9, 10, 11]: return '가을'
    return '겨울'

origin['영업일'] = pd.to_datetime(origin['영업일'])
day_str = origin['영업일'].dt.strftime('%Y-%m-%d')
origin['공휴일여부'] = day_str.isin(all_holidays).astype(int)
origin['공휴일전날여부'] = (origin['영업일'] + pd.Timedelta(days=1)).dt.strftime('%Y-%m-%d').isin(all_holidays).astype(int)
origin['요일'] = origin['영업일'].dt.day_name()
origin['계절'] = origin['영업일'].apply(get_season)
origin['연휴여부'] = day_str.isin(holiday_periods).astype(int)
origin.head()

,영업일,매출액,메인메뉴 판매건수,전통주류 판매건수,일반주류 판매건수,전체메뉴 판매건수,영수증수,평균 객단가,피크집중도,고유판매 메뉴수,연속영업일수,휴무여부,공휴일여부,공휴일전날여부,요일,계절,연휴여부
0,2024-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,Monday,여름,0
1,2024-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,Tuesday,여름,0
2,2024-07-03,124000.0,0.0,3.0,0.0,13.0,3.0,41333.0,0.00,5.0,1,0,0,0,Wednesday,여름,0
3,2024-07-04,28000.0,0.0,2.0,0.0,4.0,2.0,14000.0,0.00,3.0,2,0,0,0,Thursday,여름,0
4,2024-07-05,432500.0,1.0,8.0,4.0,34.0,8.0,54062.0,0.75,15.0,3,0,0,0,Friday,여름,0


## 7. 외부 데이터 결합

### 7.1 지하철 승하차 인원
매장 인근 2호선 신대방역의 일별 승차·하차 인원입니다.

In [27]:
subway = pd.read_csv(EXT_DIR / '신대방역_승하차.csv')
subway = subway.rename(columns={'사용일자': '영업일'})
subway['영업일'] = pd.to_datetime(subway['영업일'])
origin = pd.merge(origin, subway[['영업일', '승차총승객수', '하차총승객수']], on='영업일', how='left')

### 7.2 광고 변수
디너의여왕 체험단 리포트를 바탕으로 정리한 일별 광고 상태(`광고x`, `광고0`, `광고후 일주일` 등)에서 두 변수를 만듭니다.
* `광고_실시여부`: 광고 집행일(`광고0`)이면 1
* `광고_경과일수`: 광고가 끝난 다음 날부터 1, 2, 3 … 으로 세고, 종료일과 그 이전은 0

In [28]:
ad = pd.read_csv(EXT_DIR / '광고상태_일별.csv')
ad['영업일'] = pd.to_datetime(ad['영업일'])
origin = pd.merge(origin, ad, on='영업일', how='left')

ad_end_date = origin[origin['광고상태'] == '광고0']['영업일'].max()          # 마지막 광고 집행일
origin['광고_경과일수'] = (origin['영업일'] - ad_end_date).dt.days.apply(lambda x: x if x > 0 else 0)

active_mapping = {'광고x': 0, '광고0': 1, '광고후 일주일': 0, '광고후 이주일': 0, '광고후 삼주일': 0}
origin['광고_실시여부'] = origin['광고상태'].map(active_mapping)
origin = origin.drop(columns=['광고상태'])

### 7.3 기상특보여부
폭염·한파 특보 가운데 하나라도 발효된 날을 1로 표시합니다. 특보 자료에 날짜가 없는 날은 특보가 없던 날(0)로 봅니다.

In [29]:
heat = pd.read_csv(EXT_DIR / '폭염특보.csv')
cold = pd.read_csv(EXT_DIR / '한파특보.csv')
heat['폭염특보'] = np.where(heat['폭염특보'].astype(str).str.strip() == '없음', 0, 1)
cold['한파특보'] = np.where(cold['한파특보'].astype(str).str.strip() == '없음', 0, 1)

alert = pd.merge(heat, cold, on='일시', how='outer')
alert[['폭염특보', '한파특보']] = alert[['폭염특보', '한파특보']].fillna(0).astype(int)
alert['기상특보여부'] = np.where((alert['폭염특보'] == 1) | (alert['한파특보'] == 1), 1, 0)
alert['일시'] = pd.to_datetime(alert['일시'])

origin = pd.merge(origin, alert[['일시', '기상특보여부']], left_on='영업일', right_on='일시', how='left')
origin = origin.drop(columns=['일시'])

### 7.4 총강수량
시간별 강수량(mm)과 적설(cm)을 더해 하루 합계를 냅니다. 적설 1cm를 강수량 1mm로 간주해 단순 합산했습니다.

In [30]:
weather = pd.read_csv(EXT_DIR / '날씨_시간별.csv')
weather['일시'] = pd.to_datetime(weather['일시'])
weather['총강수량(mm)'] = weather['강수량(mm)'].fillna(0) + weather['적설(cm)'].fillna(0)

rain = weather.groupby(weather['일시'].dt.date)['총강수량(mm)'].sum().reset_index()
rain['일시'] = pd.to_datetime(rain['일시'])
origin = pd.merge(origin, rain, left_on='영업일', right_on='일시', how='left').drop(columns=['일시'])

### 7.5 일평균기온

In [31]:
temp = pd.read_excel(EXT_DIR / '일별_평균기온.xlsx')
origin = pd.merge(origin, temp, left_on='영업일', right_on='일자', how='left').drop(columns=['일자'])

### 7.6 리뷰조회수
체험단 광고로 올라간 블로그 리뷰 글의 일별 조회수 합계입니다. 광고 시작 다음 날(2024-11-25)부터 집계되며 관측 기간 마지막 날(2026-06-30)까지만 씁니다. 광고 이전 기간은 결측으로 남기고 결측 처리는 본 분석 노트북의 3장에서 합니다.

In [32]:
review = pd.read_csv(EXT_DIR / '리뷰조회수_일별.csv')
review['영업일'] = pd.to_datetime(review['영업일'])
review = review[review['영업일'] <= '2026-06-30']
origin = pd.merge(origin, review, on='영업일', how='left')
origin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 730 entries, 0 to 729
Data columns (total 25 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   영업일        730 non-null    datetime64[ns]
 1   매출액        708 non-null    float64       
 2   메인메뉴 판매건수  708 non-null    float64       
 3   전통주류 판매건수  708 non-null    float64       
 4   일반주류 판매건수  708 non-null    float64       
 5   전체메뉴 판매건수  708 non-null    float64       
 6   영수증수       708 non-null    float64       
 7   평균 객단가     708 non-null    float64       
 8   피크집중도      708 non-null    float64       
 9   고유판매 메뉴수   708 non-null    float64       
 10  연속영업일수     730 non-null    int64         
 11  휴무여부       730 non-null    int64         
 12  공휴일여부      730 non-null    int64         
 13  공휴일전날여부    730 non-null    int64         
 14  요일         730 non-null    object        
 15  계절         730 non-null    object        
 16  연휴여부       730 non-null    int64         
 1

## 8. 저장

완성된 730일 × 25개 변수 테이블을 `origin.xlsx`로 저장합니다. 이 파일이 본 분석 노트북의 입력 데이터입니다.

In [33]:
print(f"최종 테이블 크기: {origin.shape}")
origin.to_excel(OUTPUT_PATH, index=False)

최종 테이블 크기: (730, 25)
